# Zeroth-Order Optimization (ZOO) Attack on All Models

This notebook mirrors the structure of your HopSkipJump attack notebook, but
uses the **Zeroth-Order Optimization (ZOO)** attack from the Adversarial
Robustness Toolbox (ART) to generate adversarial examples against all of your
trained models.

It assumes you have helper functions available to train and return your models,
for example:

- `run_logreg("CSVs/newDataset.csv")`
- `run_neuralnet("CSVs/newDataset.csv")`
- `run_randomforest("CSVs/newDataset.csv")`
- `run_svm("CSVs/newDataset.csv")`
- `run_xgboost("CSVs/newDataset.csv")`

Each function is expected to return a tuple:

```python
(model, X_test, y_test)
```

where:
- `model` is a trained sklearn-compatible classifier (pipeline or estimator)
- `X_test` is the test feature matrix (pandas DataFrame or NumPy array)
- `y_test` is the corresponding label vector


In [23]:
# Imports

import numpy as np
import pandas as pd
import warnings

from sklearn.metrics import accuracy_score

from art.utils import check_and_transform_label_format
from art.estimators.classification import SklearnClassifier


from art.attacks.evasion import ZooAttack

# Optional: silence the StandardScaler feature-name warning
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but StandardScaler was fitted with feature names",
    category=UserWarning,
)

# IMPORTANT: Import your run_* helpers here.
# Adjust the import paths and function names to match your project.
#
# Example (update to your actual module paths):
from MachineLearning.LogReg.LogisticRegression_ML import run_best_model as run_logreg
from MachineLearning.NeuralNetworks.NeuralNet_ML import run_best_model as run_neuralnet
from MachineLearning.RandomForest.RandomForest_ML import run_best_model as run_randomforest
from MachineLearning.SVM.SVM_ML import run_best_model as run_svm
from MachineLearning.XGBoost.XGBoost_ML import run_best_model as run_xgboost


In [24]:
# Set seed for reproducibility

SEED = 42
np.random.seed(SEED)

In [25]:
# Run all models on the same dataset path.
# Each run_* function should return (model, X_test, y_test).

DATA_PATH = "CSVs/newDataset.csv"

model_runs = {}

print("Running Logistic Regression...")
logreg_model, logreg_X_test, logreg_y_test = run_logreg(DATA_PATH)
model_runs["LogisticRegression"] = (logreg_model, logreg_X_test, logreg_y_test)

print("Running Neural Net...")
nn_model, nn_X_test, nn_y_test = run_neuralnet(DATA_PATH)
model_runs["NeuralNet"] = (nn_model, nn_X_test, nn_y_test)

print("Running Random Forest...")
rf_model, rf_X_test, rf_y_test = run_randomforest(DATA_PATH)
model_runs["RandomForest"] = (rf_model, rf_X_test, rf_y_test)

print("Running SVM...")
svm_model, svm_X_test, svm_y_test = run_svm(DATA_PATH)
model_runs["SVM"] = (svm_model, svm_X_test, svm_y_test)

print("Running XGBoost...")
xgb_model, xgb_X_test, xgb_y_test = run_xgboost(DATA_PATH)
model_runs["XGBoost"] = (xgb_model, xgb_X_test, xgb_y_test)

print("\nSummary of collected models:")
for name, (model, X_test, y_test) in model_runs.items():
    print(f" - {name}: model={type(model)}, X_test shape={getattr(X_test, 'shape', None)}")

Running Logistic Regression...
Using parameters: {'clf__C': 10, 'clf__class_weight': 'balanced', 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}

Saved last run results to Results/LogRegResults\results_logreg.csv
Saved ROC data to Results/LogRegResults\roc_logreg_clean.csv (AUC = 0.936)
Saved summary (AUC + Confusion Matrix) to Results/LogRegResults\logreg_summary.csv

=== Test Set Classification Report ===
              precision    recall  f1-score   support

           0       0.96      0.94      0.95      1352
           1       0.77      0.83      0.80       347

    accuracy                           0.92      1699
   macro avg       0.87      0.89      0.87      1699
weighted avg       0.92      0.92      0.92      1699


Confusion Matrix:
         Pred 0  Pred 1
True 0    1268      84
True 1      58     289

AUC: 0.936

=== All results and summaries saved successfully ===
Running Neural Net...
Using parameters: {'hidden_layer_sizes': '128,64', 'alpha': 0.0001, 'learning_rate_

C:\Users\Jan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\training.py:183: UserWarning: [00:15:21] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [29]:
# ZOO attack on each model

zoo_kwargs = dict(
    max_iter=10,
    binary_search_steps=5,
    initial_const=1e-3,
    learning_rate=1e-2,
    nb_parallel=1,
    batch_size=1,   # leave this as 1
    targeted=False,
)

results = {}

# Limit samples per model for speed
MAX_SAMPLES = 200

for name, (model, X_test, y_test) in model_runs.items():
    print("\n=== ZOO attack on model:", name, "===")

    # Convert X_test, y_test to numpy arrays
    if hasattr(X_test, "to_numpy"):
        X_all = X_test.to_numpy().astype(np.float32)
    else:
        X_all = np.array(X_test, dtype=np.float32)

    if hasattr(y_test, "to_numpy"):
        y_all = y_test.to_numpy()
    else:
        y_all = np.array(y_test)

    # Choose up to MAX_SAMPLES random points to attack
    n = min(MAX_SAMPLES, len(X_all))
    idx = np.random.choice(len(X_all), size=n, replace=False)
    X_subset = X_all[idx]
    y_subset = y_all[idx]

    print(f"Using {n} samples for ZOO on {name}")

    # Wrap model for ART (no clip_values to avoid min >= max issues)
    art_clf = SklearnClassifier(model=model)

    clean_correct = 0
    adv_correct = 0

    # Attack samples one by one (ZOO needs batch_size=1)
    for i in range(n):
        Xi = X_subset[i:i+1]   # shape (1, n_features)
        yi = y_subset[i:i+1]

        attack = ZooAttack(classifier=art_clf, **zoo_kwargs)
        X_adv_i = attack.generate(x=Xi)

        pred_clean_i = model.predict(Xi)
        pred_adv_i = model.predict(X_adv_i)

        clean_correct += (pred_clean_i == yi).sum()
        adv_correct += (pred_adv_i == yi).sum()

    clean_acc = clean_correct / n
    adv_acc = adv_correct / n


    print(f"{name} clean acc: {clean_acc:.4f}")
    print(f"{name} adv   acc: {adv_acc:.4f}")

    results[name] = dict(clean_acc=float(clean_acc), adv_acc=float(adv_acc))

results_df = pd.DataFrame(results).T
results_df


=== ZOO attack on model: LogisticRegression ===
Using 200 samples for ZOO on LogisticRegression


ZOO: 100%|██████████| 1/1 [00:00<00:00, 142.80it/s]


LogisticRegression clean acc: 0.8900
LogisticRegression adv   acc: 0.1650

=== ZOO attack on model: NeuralNet ===
Using 200 samples for ZOO on NeuralNet


ZOO: 100%|██████████| 1/1 [00:00<00:00, 133.09it/s]


NeuralNet clean acc: 0.9500
NeuralNet adv   acc: 0.2250

=== ZOO attack on model: RandomForest ===
Using 200 samples for ZOO on RandomForest


ZOO: 100%|██████████| 1/1 [00:00<00:00,  1.72it/s]


RandomForest clean acc: 0.9400
RandomForest adv   acc: 0.9350

=== ZOO attack on model: SVM ===
Using 200 samples for ZOO on SVM


ZOO: 100%|██████████| 1/1 [00:00<00:00, 142.84it/s]

SVM clean acc: 0.9200
SVM adv   acc: 0.2200

=== ZOO attack on model: XGBoost ===
Using 200 samples for ZOO on XGBoost


TypeError: Model is not an sklearn model. Received '<class 'xgboost.sklearn.XGBClassifier'>'